# QMCPy — Acceptance-Rejection True Measure
## Based on Zhu & Dick (2014): *Discrepancy bounds for deterministic acceptance-rejection samplers*

---

### What this notebook does

This notebook introduces a new `AcceptanceRejection` **TrueMeasure** class for the **QMCPy** library.  It follows the QMCPy `AbstractTrueMeasure` interface so users can swap it into any existing QMCPy workflow where they currently use `Uniform` or `Gaussian`.

The class is built directly from the three algorithms in Zhu & Dick (2014):

| Algorithm | Class | Key idea |
|-----------|-------|----------|
| **Alg. 2** — DAR | `AcceptanceRejection` | Filter a (t,m,s)-net through `L·u ≤ ψ(x)` |
| **Alg. 3** — DAR-Real | `AcceptanceRejectionReal` | Same, after inverse Rosenblatt transform to ℝ^d |
| **Alg. 4** — DRAR | `ReducedAcceptanceRejection` | Hybrid: inversion where possible, A-R where needed |

**Why QMC instead of random A-R?**  
Standard (random) acceptance-rejection gives i.i.d. samples with star discrepancy converging at the Monte Carlo rate O(N^{-1/2}).  Replacing the driver with a genuine **(t,m,s)-net** (Sobol, Halton, …) gives a discrepancy of order **O(N^{-1/s})** — and empirically much better — per Theorem 1 of Zhu & Dick.

> **Critical requirement (M = 2^m):** The number of driver points *must* be a power of the base (here 2) so that the points form a proper (t,m,s)-net and the discrepancy bound holds.  The class enforces this automatically.


## Cell 1 — Imports

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import qmc as scipy_qmc
from scipy.stats import norm, expon
from scipy.integrate import quad
import warnings
warnings.filterwarnings('ignore')

# ── local module (will live in qmcpy/true_measure/acceptance_rejection.py) ──
import sys, os
sys.path.insert(0, '.')   # make acceptance_rejection.py importable
from acceptance_rejection import (
    AcceptanceRejection,
    AcceptanceRejectionReal,
    ReducedAcceptanceRejection,
)

plt.rcParams.update({'font.size': 12, 'figure.dpi': 110})
print("All imports successful.")
print(f"scipy version: {__import__('scipy').__version__}")
print()
print("Classes available:")
print("  AcceptanceRejection         (Algorithm 2 — DAR on [0,1]^d)")
print("  AcceptanceRejectionReal     (Algorithm 3 — DAR on ℝ^d)")
print("  ReducedAcceptanceRejection  (Algorithm 4 — DRAR)")


All imports successful.
scipy version: 1.14.1

Classes available:
  AcceptanceRejection         (Algorithm 2 — DAR on [0,1]^d)
  AcceptanceRejectionReal     (Algorithm 3 — DAR on ℝ^d)
  ReducedAcceptanceRejection  (Algorithm 4 — DRAR)


## Cell 2 — The driver: QMC vs Random

The **entire point of Zhu & Dick (2014)** is that the driver sequence must be a genuine low-discrepancy sequence, **not `np.random.rand`**.

We define two driver classes that share the same interface:
- `SobolDriver` — wraps `scipy.stats.qmc.Sobol`, always generates exactly M = 2^m points (required for the (t,m,s)-net guarantee)
- `RandomDriver` — baseline, equivalent to the standard random A-R used as comparison in the paper


In [4]:
class SobolDriver:
    """
    Genuine (t,m,s)-net driver using scipy.stats.qmc.Sobol.
    
    Always generates M = 2^m points so the (t,m,s)-net property holds.
    This is the driver that gives QMC convergence O(N^{-1/s}).
    """
    def __init__(self, dimension: int, scramble: bool = False):
        self.d        = dimension
        self.scramble = scramble
        self.mimics   = 'StdUniform'
    
    def random_base2(self, m: int) -> np.ndarray:
        """Return exactly 2^m points in [0,1)^d — the (t,m,s)-net."""
        engine = scipy_qmc.Sobol(d=self.d, scramble=self.scramble)
        return engine.random_base2(m=m)          # shape (2^m, d)
    
    def __call__(self, n: int, warn: bool = True) -> np.ndarray:
        import math
        m = int(math.ceil(math.log2(max(n, 1))))
        return self.random_base2(m=m)
    
    def __repr__(self):
        return f"SobolDriver(d={self.d}, scramble={self.scramble})"


class HaltonDriver:
    """Halton sequence driver (alternative low-discrepancy generator)."""
    def __init__(self, dimension: int, scramble: bool = False):
        self.d        = dimension
        self.scramble = scramble
        self.mimics   = 'StdUniform'
    
    def random_base2(self, m: int) -> np.ndarray:
        M = 2 ** m
        engine = scipy_qmc.Halton(d=self.d, scramble=self.scramble)
        return engine.random(M)

    def __call__(self, n: int, warn: bool = True) -> np.ndarray:
        import math
        m = int(math.ceil(math.log2(max(n, 1))))
        return self.random_base2(m=m)


class RandomDriver:
    """
    IID uniform driver — the Monte Carlo baseline (RAR in the paper).
    Use this to confirm that QMC genuinely beats random.
    """
    def __init__(self, dimension: int):
        self.d      = dimension
        self.mimics = 'StdUniform'
    
    def random_base2(self, m: int) -> np.ndarray:
        return np.random.rand(2**m, self.d)
    
    def __call__(self, n: int, warn: bool = True) -> np.ndarray:
        return np.random.rand(n, self.d)


# ── quick sanity check ──────────────────────────────────────────────────────
s = SobolDriver(dimension=3)
pts = s.random_base2(m=3)   # 8 points in 3 dims
print(f"SobolDriver random_base2(m=3): shape = {pts.shape}")
print("First 4 rows:")
print(np.round(pts[:4], 4))
print()
print("Note: M=2^m is REQUIRED for (t,m,s)-net properties (Theorem 1, Zhu & Dick).")


SobolDriver random_base2(m=3): shape = (8, 3)
First 4 rows:
[[0.   0.   0.  ]
 [0.5  0.5  0.5 ]
 [0.75 0.25 0.25]
 [0.25 0.75 0.75]]

Note: M=2^m is REQUIRED for (t,m,s)-net properties (Theorem 1, Zhu & Dick).


## Cell 3 — Algorithm 2 (DAR): basic usage

`AcceptanceRejection` is the QMCPy-style interface.  It behaves like any other
TrueMeasure: construct it, call `gen_samples(n)`, get an `(n, d)` array.

**Example:** target density  `ψ(x) = 2 x₁`  on `[0,1]²`  
- L = 2  (supremum)  
- C = ∫₀¹∫₀¹ 2x₁ dx₁dx₂ = 1  (normalisation constant)  
- Acceptance rate = C/L = 0.5  
- Expected mean of x₁ = ∫₀¹ x₁ · 2x₁ dx₁ = 2/3 ≈ 0.6667  
